**Human Movement Intelligence | Biomechanical Load Estimation | Presented by: Reza Saadatyar (2026)**

**03 | Full-Cohort Numeric and Temporal Validation**

![Full-cohort numerical and original time-coordinate workflow](../../images/03_consolidated_workflow.svg)

**Research objective:** Verify source-linked numerical IMU and FE signals across the complete original Essex dataset and match original FE time coordinates to observed IMU indices. The analytical unit remains an original movement trial.

**Illustrative mapped positions:** The original time vector for an observed trial maps the eleven FE values to zero-based IMU positions `[0, 9, 19, 29, 39, 49, 59, 69, 79, 89, 99]`, not a presumed evenly-spaced `[0, 10, ..., 100]` grid.

**Scope and limits:** Numeric channel validation, source identities, native time coordinates and FE missingness only. No interpolation, synthetic targets, full numeric data export, physical-unit validation, participant-disjoint regression evaluation or model training. The observations above describe the previous complete-cohort run.

**Execution:** The simplified cells have not been rerun on the full dataset. Numerical findings described here refer to the previously completed run. Each notebook reads the current upstream files; re-execution replaces its own output files. No historical reports, hash comparisons or PASSED gates are required.


**00. Colab Repository Setup**


In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}

/content/human-movement-intelligence


**01. Imports and Research Environment**

Use the published source loader, original numeric audit, native coordinate mapping and standard plotting/output helpers.


In [2]:
# =================== 01. Imports and Research Environment ====================

from python_modules.core.colab_setup import setup_colab

root = setup_colab("biomechanical_load_estimation")
from importlib.metadata import version
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

plot_style(
    font_family="DejaVu Sans",
    title_size=12.5,
    title_weight="normal",
    label_size=11.5,
    label_weight="normal",
    tick_size=11,
    legend_size=10,
    figure_dpi=100,
    figure_size=(9, 4),
)
from core.output_download import download_outputs_from_colab
from essex_figshare import (
    ARTICLE_ID,
    ARTICLE_VERSION,
    inspect_matlab_archive,
    extract_matlab_files,
)
from essex_timebase_audit import EXPECTED_CHANNELS, audit_original_source
from essex_temporal_mapping import audit_temporal_mapping

study_dir = root / "biomechanical_load_estimation"
raw_dir = root / "data" / "raw" / "essex_femoral_neck_loading"
mat_dir = raw_dir / "matlab_records"
output_dir = study_dir / "output"
raw_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)
print("Original Essex source:", ARTICLE_ID, "| version:", ARTICLE_VERSION)
print("Stage 03 outputs:", output_dir)

from essex_figshare import ensure_matlab_archive

RuntimeError: Local tracked/untracked changes exist. Save them before synchronising; no local files were changed.

**02. Pinned Python MATLAB Table Reader**

Keep the tested `mat-io==1.0.1` decoder version and report it before inspecting original MATLAB table values.


In [ ]:
# =================== 02. Pinned Python MATLAB Table Reader ===================

%pip -q install --only-binary=:all: --no-deps "mat-io==1.0.1"
from matio import load_from_mat

print("Python MATLAB reader:", version("mat-io"))

**03. Current Dataset and Trial Registry**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# 03. Current Dataset and Trial Registry

source_zip = ensure_matlab_archive(raw_dir)
archive_inventory = inspect_matlab_archive(source_zip)
mat_files = extract_matlab_files(source_zip, mat_dir, archive_inventory)
registry = pd.read_csv(
    output_dir / "01_original_record_registry.csv",
    dtype={
        "subject_id": str,
        "source_file": str,
        "local_record_key": str,
        "task": str,
        "updated_trial_id": str,
        "fe_representation": str,
    },
)
print("Trials:", len(registry), "| participants:", registry["subject_id"].nunique())

**04. Original IMU and FE Column Contract**

Verify 42 original sensor channels, the original FE `e1/e3` labels and finite pilot schema data without inferring measurement units.


In [ ]:
# ================== 04. Original IMU and FE Column Contract ==================

schema = pd.read_csv(output_dir / "02_python_column_schema.csv")
imu_schema = schema.loc[schema["modality"] == "IMU"]
fe_schema = schema.loc[schema["modality"] == "FE"]
imu_names = set(imu_schema["original_column_name"])
fe_names = set(fe_schema["original_column_name"])
channel_contract = pd.DataFrame(
    [
        {
            "original_column_name": label,
            "signal_component": label.rsplit("_", 1)[0],
            "sensor_code": label.rsplit("_", 1)[1],
        }
        for label in EXPECTED_CHANNELS
    ]
)
channel_contract.to_csv(output_dir / "03_imu_channel_contract.csv", index=False)
display(channel_contract.head(14))
print("Original numeric IMU channel names:", len(channel_contract))

**05. Complete Numerical Record Audit**

Use the source-linked `audit_original_source` function to inspect 1,897 records, IMU channels, FE values, reader failures and the original time axes.


In [ ]:
# ==================== 05. Complete Numerical Record Audit ====================

record_audit, reader_errors, example = audit_original_source(
    load_from_mat, mat_dir, registry
)
record_audit.to_csv(output_dir / "03_original_numeric_record_audit.csv", index=False)
reader_errors.to_csv(output_dir / "03_reader_errors.csv", index=False)
print("Full original source records audited:", len(record_audit))
print("Files with reader errors:", len(reader_errors))
display(
    record_audit[
        [
            "subject_id",
            "task",
            "imu_status",
            "fe_status",
            "imu_timestep_status",
            "fe_timestep_status",
            "time_alignment",
        ]
    ].head(20)
)

**06. Numeric Coverage across Participants and Activities**

Summarise validated numerical IMU/FE coverage and original FE-empty status for each participant and movement task.


In [ ]:
# ========== 06. Numeric Coverage across Participants and Activities ==========


def coverage(frame, by):
    return (
        frame.groupby(by)
        .agg(
            source_records=("local_record_key", "size"),
            valid_identities=("identity_valid", "sum"),
            numeric_imu=("imu_status", lambda s: s.eq("VALID_CHANNELS").sum()),
            numeric_fe=("fe_status", lambda s: s.eq("VALID_STRAIN_COLUMNS").sum()),
            source_fe_empty=("fe_status", lambda s: s.eq("EXPECTED_EMPTY").sum()),
            time_grid_matches=(
                "time_alignment",
                lambda s: s.eq("NUMERIC_TIME_GRID_MATCH").sum(),
            ),
        )
        .reset_index()
    )


participant_coverage = coverage(record_audit, "subject_id")
activity_coverage = coverage(record_audit, "task")
participant_coverage.to_csv(
    output_dir / "03_participant_numeric_coverage.csv", index=False
)
activity_coverage.to_csv(output_dir / "03_activity_numeric_coverage.csv", index=False)
display(activity_coverage)
print("Participant rows:", len(participant_coverage))

**07. Fixed-Index Timebase Comparison**

Preserve and diagnose the limited `IMU[::10]` comparison and observed source time-axis metadata, rather than assuming it proves that valid matching is impossible.


In [ ]:
# ==================== 07. Fixed-Index Timebase Comparison ====================

alignment_counts = (
    record_audit.groupby(["fe_status", "time_alignment"], dropna=False)
    .size()
    .rename("records")
    .reset_index()
)
alignment_counts.to_csv(output_dir / "03_time_alignment_status.csv", index=False)
time_cols = [
    "local_record_key",
    "subject_id",
    "task",
    "imu_timestep_status",
    "fe_timestep_status",
    "fe_time_entry_class",
    "imu_t0",
    "imu_tlast",
    "imu_tstep",
    "fe_t0",
    "fe_tlast",
    "fe_tstep",
    "time_alignment",
    "time_error",
]
for col in time_cols:
    if col not in record_audit:
        record_audit[col] = np.nan
time_examples = record_audit.loc[:, time_cols].copy()
time_examples.to_csv(output_dir / "03_timebase_metadata.csv", index=False)
display(alignment_counts)
print("FE and IMU time vectors must agree before paired export.")

**08. Native FE-to-IMU Time-Coordinate Mapping**

Match the original source FE `timestep` values directly against 101-point IMU coordinates and record unique zero-based sample indices for every source trial.


In [ ]:
# =============== 08. Native FE-to-IMU Time-Coordinate Mapping ================

mapping, fe_empty, temporal_reader_errors, source_example = audit_temporal_mapping(
    load_from_mat, mat_dir, registry
)
mapping.to_csv(output_dir / "03_original_time_coordinate_mapping.csv", index=False)
fe_empty.to_csv(output_dir / "03_source_fe_empty_representations.csv", index=False)
temporal_reader_errors.to_csv(output_dir / "03_source_reader_errors.csv", index=False)
print("All original movement records:", len(mapping))
print("Original FE-empty representations inspected:", len(fe_empty))
print("MAT reader errors:", len(temporal_reader_errors))
display(
    mapping[
        [
            "source_file",
            "task",
            "fe_channel_status",
            "mapping_status",
            "imu_position_indices",
        ]
    ].head(20)
)

**09. Source FE Missingness and Reader Empty Classes**

Reconcile all 168 source-declared FE-empty records, including reader-specific canonical empty and empty ndarray representations, without converting them into strain targets.


In [ ]:
# ============ 09. Source FE Missingness and Reader Empty Classes =============

missingness = (
    fe_empty.groupby(
        [
            "python_class",
            "python_shape",
            "empty_representation_status",
            "column_names",
            "fe_numeric_columns_finite",
        ],
        dropna=False,
    )
    .size()
    .rename("source_records")
    .reset_index()
)
missingness.to_csv(output_dir / "03_fe_missingness_types.csv", index=False)
display(missingness)
print("Total source-declared FE empties:", missingness["source_records"].sum())

**10. Original Mapping Signatures and Cohort Coverage**

Aggregate exact coordinate matches, source-empty FE cases and mapping indices; report outcomes separately for participants and activity categories.


In [ ]:
# ============ 10. Original Mapping Signatures and Cohort Coverage ============

mapping_signatures = (
    mapping.groupby(["mapping_status", "consecutive_index_steps"], dropna=False)
    .size()
    .rename("source_records")
    .reset_index()
)
mapping_signatures.to_csv(output_dir / "03_time_mapping_signatures.csv", index=False)
display(mapping_signatures.head(30))
print("Original mapping status distribution:")
print(mapping["mapping_status"].value_counts())


def group_coverage(source, by):
    return (
        source.groupby(by)
        .agg(
            original_records=("local_record_key", "size"),
            source_empty_fe=("originally_empty_fe", "sum"),
            coordinate_mappings=(
                "mapping_status",
                lambda s: s.isin(
                    [
                        "EXACT_TIME_COORDINATE_MAPPING",
                        "TOLERANT_TIME_COORDINATE_MAPPING",
                    ]
                ).sum(),
            ),
            unresolved_time=(
                "mapping_status",
                lambda s: s.eq("TIME_VALUES_NOT_AVAILABLE").sum(),
            ),
        )
        .reset_index()
    )


participant_report = group_coverage(mapping, "subject_id")
activity_report = group_coverage(mapping, "task")
participant_report.to_csv(
    output_dir / "03_participant_coordinate_coverage.csv", index=False
)
activity_report.to_csv(output_dir / "03_activity_coordinate_coverage.csv", index=False)
display(activity_report)
print("Participants reported:", len(participant_report))

**11. Full Numeric Coverage and Source Signal Figures**

Show per-activity finite channels, fixed-grid classifications and authentic numerical IMU and e1/e3 traces. Do not display predicted clinical strain.


In [ ]:
# ============ 11. Full Numeric Coverage and Source Signal Figures ============

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5.1),
    gridspec_kw={"width_ratios": [1.7, 1.25, 1.0]},
    constrained_layout=True,
)
tasks = activity_coverage["task"]
positions = np.arange(len(tasks))
axes[0].bar(
    positions - 0.16,
    activity_coverage["numeric_imu"],
    width=0.31,
    label="42 IMU channels",
)
axes[0].bar(
    positions + 0.16, activity_coverage["numeric_fe"], width=0.31, label="FE e1 and e3"
)
axes[0].set_xticks(positions)
axes[0].set_xticklabels(
    [name.replace("Hopping", " Hopping").replace("Jumps", " Jumps") for name in tasks],
    rotation=25,
    ha="right",
)
axes[0].set_title("Numeric source coverage by activity")
axes[0].set_ylabel("Original records")
axes[0].legend(frameon=False)
axes[0].grid(axis="y", alpha=0.2)
statuses = record_audit.groupby("time_alignment").size().sort_values(ascending=True)
yy = np.arange(len(statuses))
axes[1].barh(yy, statuses.values, height=0.34)
axes[1].set_yticks(yy)
axes[1].set_yticklabels([s.replace("_", " ").title() for s in statuses.index])
axes[1].set_title("IMU–FE time-grid classifications")
axes[1].set_xlabel("Original records")
axes[1].grid(axis="x", alpha=0.2)
if example is not None and example["imu_time"] is not None:
    axes[2].plot(example["imu_time"], np.zeros(101), ".", label="IMU grid")
    if example["fe_time"] is not None:
        axes[2].plot(
            example["fe_time"], np.ones(11), "o", markersize=4, label="FE grid"
        )
    axes[2].set_yticks([0, 1])
    axes[2].set_yticklabels(["IMU", "FE"])
    axes[2].set_title("Original source time coordinates")
    axes[2].set_xlabel("Original time coordinate")
    axes[2].legend(frameon=False)
else:
    axes[2].text(
        0.5,
        0.5,
        "No numeric source time axis available",
        ha="center",
        va="center",
        transform=axes[2].transAxes,
    )
    axes[2].set_title("Original time-axis availability")
    axes[2].set_xticks([])
    axes[2].set_yticks([])
save_figure(fig, output_dir / "03_numeric_alignment_diagnostics.png")
plt.show()
fig2, axs = plt.subplots(1, 2, figsize=(13.5, 4.5), constrained_layout=True)
if example is not None:
    axs[0].plot(
        np.arange(len(example["imu_signal"])), example["imu_signal"], linewidth=1.4
    )
    axs[0].set_title("Original IMU: " + example["imu_name"])
    axs[0].set_ylabel("Source numerical value")
    axs[0].set_xlabel("IMU sample index")
    axs[1].plot(np.arange(len(example["e1"])), example["e1"], label="e1", linewidth=1.4)
    axs[1].plot(np.arange(len(example["e3"])), example["e3"], label="e3", linewidth=1.4)
    axs[1].set_title("Original FE strain columns")
    axs[1].set_xlabel("FE sample index")
    axs[1].set_ylabel("Source strain value")
    axs[1].legend(frameon=False)
else:
    for axis in axs:
        axis.text(
            0.5,
            0.5,
            "No validated numeric example",
            ha="center",
            va="center",
            transform=axis.transAxes,
        )
        axis.set_xticks([])
        axis.set_yticks([])
save_figure(fig2, output_dir / "03_original_numeric_examples.png")
plt.show()

**12. Original Time-Coordinate and FE Coverage Figures**

Use observed source coordinates and mapped index positions to visualise FE/IMU mapping and separately chart original FE-empty representation types.


In [ ]:
# =========== 12. Original Time-Coordinate and FE Coverage Figures ============

fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5.1),
    gridspec_kw={"width_ratios": [1.6, 1.1, 1.1]},
    constrained_layout=True,
)
pos = np.arange(len(activity_report))
axes[0].bar(
    pos - 0.17,
    activity_report["coordinate_mappings"],
    width=0.33,
    label="Mapped FE time vectors",
)
axes[0].bar(
    pos + 0.17, activity_report["source_empty_fe"], width=0.33, label="Source-empty FE"
)
axes[0].set_xticks(pos)
axes[0].set_xticklabels(
    activity_report["task"].str.replace("Hopping", " Hopping", regex=False),
    rotation=26,
    ha="right",
)
axes[0].set_title("Original FE coordinate coverage")
axes[0].set_ylabel("Movement records")
axes[0].legend(frameon=False)
axes[0].grid(axis="y", alpha=0.2)
state_counts = mapping["mapping_status"].value_counts().sort_values()
axes[1].barh(np.arange(len(state_counts)), state_counts.values, height=0.32)
axes[1].set_yticks(np.arange(len(state_counts)))
axes[1].set_yticklabels([s.replace("_", " ").title() for s in state_counts.index])
axes[1].set_title("Original mapping classifications")
axes[1].set_xlabel("Movement records")
axes[1].grid(axis="x", alpha=0.2)
class_counts = fe_empty["empty_representation_status"].value_counts()
axes[2].barh(np.arange(len(class_counts)), class_counts.values, height=0.32)
axes[2].set_yticks(np.arange(len(class_counts)))
axes[2].set_yticklabels([s.replace("_", " ").title() for s in class_counts.index])
axes[2].set_title("Source-empty FE representations")
axes[2].set_xlabel("Original source-empty records")
axes[2].grid(axis="x", alpha=0.2)
save_figure(fig, output_dir / "03_temporal_mapping_diagnostics.png")
plt.show()
fig2, axs = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
if source_example is not None:
    actual_imu = source_example["imu_time"]
    actual_fe = source_example["fe_time"]
    mapped = np.asarray(source_example["imu_indices"])
    axs[0].plot(
        np.arange(len(actual_imu)), actual_imu, linewidth=1.2, label="Original IMU time"
    )
    axs[0].scatter(mapped, actual_fe, s=24, label="Original FE time", zorder=3)
    axs[0].set_title("Native FE coordinates on original IMU positions")
    axs[0].set_xlabel("Original IMU sample index (zero-based)")
    axs[0].set_ylabel("Original time coordinate")
    axs[0].legend(frameon=False)
    axs[1].plot(np.arange(len(actual_fe)), mapped, "o-", linewidth=1.2)
    axs[1].set_title("Source FE-to-IMU sample mapping")
    axs[1].set_xlabel("Original FE sample index")
    axs[1].set_ylabel("Matched IMU sample index")
else:
    for axis in axs:
        axis.text(
            0.5,
            0.5,
            "No unique original time mapping",
            ha="center",
            va="center",
            transform=axis.transAxes,
        )
        axis.set_xticks([])
        axis.set_yticks([])
save_figure(fig2, output_dir / "03_original_coordinate_example.png")
plt.show()

**13. Results Summary and Output Download**

Read the current data and display the observed results. Re-running this section replaces its own saved outputs.


In [ ]:
# 13. Results Summary and Output Download

display(
    record_audit.groupby(["imu_status", "fe_status"], dropna=False)
    .size()
    .rename("trials")
    .reset_index()
)
print("Current temporal mapping: 03_original_time_coordinate_mapping.csv")
download_outputs_from_colab(folder="biomechanical_load_estimation/output", stage="03")

**14. Scientific Interpretation and Next Research Boundary**

The sample source-derived FE index signature is `[0, 9, 19, 29, 39, 49, 59, 69, 79, 89, 99]`. The earlier fixed-grid mismatch is therefore a **specific sampling-index assumption failure**, not a failure of the original FE–IMU time relationship.

**Limitations:** No inference of physical strain units, imputation for source-missing FE, full raw-data conversion, clinical deployment, subject-independent regression or predictive model training. Complete coordinate matching does not alone establish regression readiness.

**Next independent research topic:** Governed dataset construction and participant-level evaluation design once target semantics and unit provenance have been assessed.

**Next step:** Notebook 04 uses the current trial registry and time-coordinate mapping to build paired arrays.
